GQA（Group Query Attention）的优点：效果损失小，推理的时候可以加速（来自于kvcache小，内存取数少）。

仔细阅读 MHA, MQA 和 GQA的区别，就会发现 MHA 和 MQA 都是 GQA 的特殊表达形式

三者可以用同一套代码，只需要修改【GQA】代码里面的 nums_key_value_head 参数就可

nums_key_value_head 设置等于 1 就是 MQA

nums_key_value_head 设置等于 nums_head 就是 MHA

备注：以下代码省略了 attention_dropout attention_mask等情况的处理，真实实现过程中需要考虑。

In [1]:
import torch
import torch.nn as nn
import math

In [5]:
# 忽略了 attention_mask, attention_dropout; 
class GroupQueryAttention(nn.Module):
    def __init__(self, hidden_dim, nums_heads, nums_key_value_head):
        super().__init__()
        assert hidden_dim % nums_heads == 0 
        assert nums_heads % nums_key_value_head == 0

        self.hidden_dim = hidden_dim
        self.nums_heads = nums_heads
        self.nums_key_value_head = nums_key_value_head
        self.head_dim = hidden_dim // nums_heads

        # initialisation q k v o
        self.q_proj = nn.Linear(hidden_dim, nums_heads * self.head_dim)
        # k v out shape (nums_key_value_head * head_dim)
        self.k_proj = nn.Linear(hidden_dim, nums_key_value_head * self.head_dim)
        self.v_proj = nn.Linear(hidden_dim, nums_key_value_head * self.head_dim)

        self.o_proj = nn.Linear(hidden_dim, hidden_dim) # input_size nums_head * head_dim

    def forward(self, X, attention_mask=None):
        # X shape (bath, seq, hidden_dim)
        batch_size, seq, _ = X.size()

        # qkv projection
        q = self.q_proj(X) # (batch, seq, hidden_dim)
        k = self.k_proj(X) 
        v = self.v_proj(X) # (batch, seq,  nums_key_value_head * self.head_dim)

        # attention_weight aim's shape is (batch, nums_heads, seq, seq)
        q = q.view(batch_size, seq, self.nums_heads, self.head_dim)
        k = k.view(batch_size, seq, self.nums_key_value_head, self.head_dim)
        v = v.view(batch_size, seq, self.nums_key_value_head, self.head_dim)

        # focuse on nums_head and nums_key_value_head
        q = q.transpose(1, 2) # (b, nums_head, seq, head_dim)
        k = k.transpose(1, 2) # (b, nums_key_value_head, seq, head_dim)
        v = v.transpose(1, 2) # (b, nums_key_value_head, seq, head_dim)

        # k v repeat to make sure we can calcualte the attentions
        # we make them as the same dimension as we have done in MAH
        k = k.repeat_interleave(self.nums_heads // self.nums_key_value_head, dim=1)
        v = v.repeat_interleave(self.nums_heads // self.nums_key_value_head, dim=1)

        attention_score = (q @ k.transpose(2, 3)) / math.sqrt(self.head_dim)

        attention_weight = torch.softmax(attention_score, dim = -1)
        ## (attention_mask we ingore here) Please see the MAH before. It is the same

        output = attention_weight @ v # (b, nums_head, seq, head_dim)

        # output project become (b, seq, hidden_dim)
        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq, -1)
        final_output = self.o_proj(output)

        return final_output


In [6]:
x = torch.rand(3, 2, 128)
net = GroupQueryAttention(128, 8, 4)

In [7]:
net(x).shape

torch.Size([3, 2, 128])